### Update projected starting lineups

In [11]:
# from MODELS.scrapStarting import NBADailyLineups

# scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
# scraper.getDict()  # Scrape the lineups
# scraper.updateTeamInfo()  # Update teamInfo.py

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [13]:
# Load split NGBoost models (mean, variance, calibration factor, and isotonic calibrator)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')

model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 4.5


### Load Player Data and Bookmaker Data

In [14]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_34686/1321450873.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Underdog,player_points,Karl-Anthony Towns,Over,21.5,-137,2025-11-22,2025-11-22T21:36:52Z
1,Underdog,player_points,Karl-Anthony Towns,Under,21.5,-137,2025-11-22,2025-11-22T21:36:52Z
2,Underdog,player_points,Desmond Bane,Over,20.5,-137,2025-11-22,2025-11-22T21:36:52Z
3,Underdog,player_points,Desmond Bane,Under,20.5,-137,2025-11-22,2025-11-22T21:36:52Z
4,Underdog,player_points,Franz Wagner,Over,23.5,-137,2025-11-22,2025-11-22T21:36:52Z


### Top EVs for single bets

In [15]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

singleBets = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)



singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION', 'SIDE','ODDS','RECOMMENDATION', 'EV%', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets...
Pre-computing predictions for 94 unique players...
Error getting prediction for Keegan Murray: float division by zero


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV%,KELLY_FRACTION,SIGMA FLAG
650,Coby White,BetRivers,22.5,25.74,Over,120,0,57.98,0.483,Med
636,Ayo Dosunmu,BetRivers,15.5,19.48,Over,114,0,57.15,0.501,High
700,Jalen Duren,BetRivers,18.5,22.25,Over,107,0,46.38,0.433,High
24,Miles McBride,FanDuel,8.5,11.82,Over,102,0,41.49,0.407,High
680,Ausar Thompson,BetRivers,10.5,13.38,Over,104,0,39.06,0.376,High


## Top EVs for 2 leg bets

### Underdog picks

In [16]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 65 players...
Processing 58 players...
Generated 1535 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
903,Coby White,Cameron Johnson,20.5,13.5,25.74,10.05,0.825,0.728,over,under,0,76.44,0.382,Med,Med
281,Jonathan Isaac,Ayo Dosunmu,3.5,14.5,6.04,19.48,0.711,0.783,over,over,0,63.75,0.319,Low,High
27,Karl-Anthony Towns,Myles Turner,21.5,15.5,24.58,12.32,0.669,0.697,over,under,0,37.16,0.186,High,High
1335,Duncan Robinson,Santi Aldama,10.5,17.5,13.07,14.94,0.664,0.660,over,under,0,28.89,0.144,High,High
188,Jordan Clarkson,Bobby Portis,9.5,15.5,11.95,13.15,0.658,0.651,over,under,0,25.98,0.130,High,High
374,Landry Shamet,Jeremiah Fears,8.5,16.5,10.33,18.78,0.640,0.640,over,over,0,20.33,0.102,Med,High
671,Nickeil Alexander-Walker,Zach LaVine,18.5,18.5,20.56,20.85,0.619,0.639,over,over,0,16.24,0.081,High,High
867,Alex Sarr,DeMar DeRozan,17.5,17.5,19.36,15.35,0.610,0.628,over,under,0,12.76,0.064,High,High
358,Goga Bitadze,Jamal Murray,4.5,23.5,5.57,25.47,0.605,0.607,over,over,0,7.91,0.040,Low,High
1279,Caris LeVert,Dennis Schröder,7.5,11.5,8.82,13.25,0.597,0.607,over,over,0,6.48,0.032,Med,High


### Prizepicks picks

In [17]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 89 players...


Error getting prediction for Keegan Murray: float division by zero
Processing 78 players...
Generated 2784 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV%,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
1657,Coby White,Jalen Duren,20.5,17.5,25.74,22.25,over,over,0.825,0.755,0.6102,0.247,0.177,0.289,83.07,0.415,1,5.61,6.87,Med,High,"(14.7, 36.7)","(8.8, 35.7)",0.05,0,83.1
1861,Ayo Dosunmu,Peyton Watson,14.5,12.5,19.48,9.44,over,under,0.783,0.753,0.5777,0.205,0.175,0.255,73.31,0.367,0,6.36,4.49,High,Low,"(7.0, 31.9)","(0.6, 18.2)",0.05,0,73.3
2396,Tobias Harris,Cameron Johnson,10.5,13.5,14.89,10.05,over,under,0.751,0.728,0.5353,0.173,0.150,0.212,60.59,0.303,0,6.50,5.70,High,Med,"(2.2, 27.6)","(0.0, 21.2)",0.05,0,60.6
915,Jonathan Isaac,Ausar Thompson,3.5,9.5,6.04,13.38,over,over,0.711,0.738,0.5141,0.133,0.160,0.190,54.24,0.271,0,4.57,6.09,Low,High,"(0.0, 15.0)","(1.4, 25.3)",0.05,0,54.2
782,Miles McBride,Myles Turner,8.5,15.5,11.82,12.32,over,under,0.700,0.697,0.4786,0.122,0.119,0.154,43.57,0.218,0,6.32,6.15,High,High,"(0.0, 24.2)","(0.3, 24.4)",0.05,0,43.6
196,Karl-Anthony Towns,Jock Landale,21.5,8.5,24.58,10.98,over,over,0.669,0.677,0.4436,0.091,0.098,0.119,33.09,0.165,0,7.05,5.42,High,Med,"(10.8, 38.4)","(0.4, 21.6)",0.05,0,33.1
969,Jalen Johnson,Duncan Robinson,22.5,10.5,25.33,13.07,over,over,0.666,0.664,0.4331,0.088,0.086,0.108,29.94,0.150,0,6.61,6.07,High,High,"(12.4, 38.3)","(1.2, 25.0)",0.05,0,29.9
658,Jordan Clarkson,Santi Aldama,9.5,17.5,11.95,14.94,over,under,0.658,0.660,0.4260,0.080,0.082,0.101,27.81,0.139,0,6.01,6.19,High,High,"(0.2, 23.7)","(2.8, 27.1)",0.05,0,27.8
1192,Jeremiah Fears,Bobby Portis,16.5,15.5,18.78,13.15,over,under,0.640,0.651,0.4082,0.062,0.073,0.082,22.45,0.112,0,6.38,6.06,High,High,"(6.3, 31.3)","(1.3, 25.0)",0.05,0,22.5
2628,D'Angelo Russell,Zach LaVine,12.5,18.5,14.79,20.85,over,over,0.630,0.639,0.3946,0.052,0.061,0.068,18.37,0.092,0,6.91,6.59,High,High,"(1.2, 28.3)","(7.9, 33.8)",0.05,0,18.4


## 3 leg parlay

### Underdog picks

In [18]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogTrios = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 65 players...
Processing 58 players...
Generated 30155 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
21618,Coby White,Ayo Dosunmu,Cameron Johnson,20.5,14.5,13.5,25.74,19.48,10.05,0.825,0.783,0.728,over,over,under,0,153.83,0.308,Med,High,Med
8360,Jonathan Isaac,Myles Turner,Santi Aldama,3.5,15.5,17.5,6.04,12.32,14.94,0.711,0.697,0.660,over,under,under,0,76.80,0.154,Low,High,High
127,Karl-Anthony Towns,Jordan Clarkson,Duncan Robinson,21.5,9.5,10.5,24.58,11.95,13.07,0.669,0.658,0.664,over,over,over,0,57.89,0.116,High,High,High
10356,Landry Shamet,Jeremiah Fears,Bobby Portis,8.5,16.5,15.5,10.33,18.78,13.15,0.640,0.640,0.651,over,over,under,0,43.88,0.088,Med,High,High
17438,Nickeil Alexander-Walker,Zach LaVine,DeMar DeRozan,18.5,18.5,17.5,20.56,20.85,15.35,0.619,0.639,0.628,over,over,under,0,34.18,0.068,High,High,High


### Prizepicks picks

In [19]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=15)


triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 89 players...
Error getting prediction for Keegan Murray: float division by zero
Processing 78 players...
Generated 74300 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
54720,Coby White,Ayo Dosunmu,Jalen Duren,20.5,14.5,17.5,25.74,19.48,22.25,0.825,0.783,0.755,over,over,over,1,163.38,0.327,Med,High,High
70134,Tobias Harris,Ausar Thompson,Peyton Watson,10.5,9.5,12.5,14.89,13.38,9.44,0.751,0.738,0.753,over,over,under,0,125.07,0.250,High,High,Low
33478,Jonathan Isaac,Myles Turner,Cameron Johnson,3.5,15.5,13.5,6.04,12.32,10.05,0.711,0.697,0.728,over,under,under,0,94.81,0.190,Low,High,Med
6193,Karl-Anthony Towns,Miles McBride,Jock Landale,21.5,8.5,8.5,24.58,11.82,10.98,0.669,0.700,0.677,over,over,over,0,71.22,0.142,High,High,Med
35423,Jalen Johnson,Duncan Robinson,Santi Aldama,22.5,10.5,17.5,25.33,13.07,14.94,0.666,0.664,0.660,over,over,under,0,57.61,0.115,High,High,High


In [20]:
# df = playerScoring('Trey Murphy III', s26, current_date, teamStarPlayer, projectedStartingFive)
# playerContext('Trey Murphy III', s26, current_date, projectedStartingFive, mainStartingFive, teamStarPlayer)